# T4 · The relationship register

## Part A · The six relationships

The model contains one fact table and five dimensions. Every relationship is dimension (one) to fact (many), with single-direction filtering. The origin-city path is active; the destination-city path is inactive because both city keys point to the same `DimCity` table.

In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path.cwd()
if not (DATA_DIR / "FactShipment.csv").exists():
    DATA_DIR = Path("Week04_Warehouse")
fact = pd.read_csv(DATA_DIR / "FactShipment.csv")
dimensions = {
    "DimDate": pd.read_csv(DATA_DIR / "DimDate.csv"),
    "DimCity": pd.read_csv(DATA_DIR / "DimCity.csv"),
    "DimRoute": pd.read_csv(DATA_DIR / "DimRoute.csv"),
    "DimMode": pd.read_csv(DATA_DIR / "DimMode.csv"),
    "DimStatus": pd.read_csv(DATA_DIR / "DimStatus.csv"),
}

assert fact.WeightKG.sum() == 578_650
assert fact.FreightCostPKR.sum() == 4_808_000
print(f"FactShipment rows: {len(fact):,}")
print(f"Total weight: {fact.WeightKG.sum():,} kg")
print(f"Total freight cost: PKR {fact.FreightCostPKR.sum():,}")

FactShipment rows: 34
Total weight: 578,650 kg
Total freight cost: PKR 4,808,000


## Relationship register

| # | One side (dimension) | Many side (fact) | Cardinality | Cross-filter | Active? | Why this relationship exists |
|---:|---|---|---|---|---|---|
| 1 | `DimDate[DateKey]` | `FactShipment[DateKey]` | `* : 1` | Single | Yes | Filters shipments by shipment date. |
| 2 | `DimCity[CityKey]` | `FactShipment[OriginCityKey]` | `* : 1` | Single | Yes | Filters shipments by their origin city. |
| 3 | `DimCity[CityKey]` | `FactShipment[DestCityKey]` | `* : 1` | Single | No | Destination-city role; kept inactive to avoid two active paths between `DimCity` and `FactShipment`. |
| 4 | `DimRoute[RouteKey]` | `FactShipment[RouteKey]` | `* : 1` | Single | Yes | Filters shipments by route and corridor. |
| 5 | `DimMode[ModeKey]` | `FactShipment[ModeKey]` | `* : 1` | Single | Yes | Filters shipments by transport mode. |
| 6 | `DimStatus[StatusKey]` | `FactShipment[StatusKey]` | `* : 1` | Single | Yes | Filters shipments by delivery status. |

In [2]:
relationships = pd.DataFrame([
    ["DimDate[DateKey]", "FactShipment[DateKey]", "* : 1", "Single", "Yes", "Filters shipments by shipment date."],
    ["DimCity[CityKey]", "FactShipment[OriginCityKey]", "* : 1", "Single", "Yes", "Filters shipments by their origin city."],
    ["DimCity[CityKey]", "FactShipment[DestCityKey]", "* : 1", "Single", "No", "Destination-city role; kept inactive to avoid two active paths between DimCity and FactShipment."],
    ["DimRoute[RouteKey]", "FactShipment[RouteKey]", "* : 1", "Single", "Yes", "Filters shipments by route and corridor."],
    ["DimMode[ModeKey]", "FactShipment[ModeKey]", "* : 1", "Single", "Yes", "Filters shipments by transport mode."],
    ["DimStatus[StatusKey]", "FactShipment[StatusKey]", "* : 1", "Single", "Yes", "Filters shipments by delivery status."],
], columns=["One side (dimension)", "Many side (fact)", "Cardinality", "Cross-filter", "Active?", "Why this relationship exists"])
relationships.index = range(1, len(relationships) + 1)
relationships.index.name = "#"
relationships

,One side (dimension),Many side (fact),Cardinality,Cross-filter,Active?,Why this relationship exists
#,,,,,,
1,DimDate[DateKey],FactShipment[DateKey],* : 1,Single,Yes,Filters shipments by shipment date.
2,DimCity[CityKey],FactShipment[OriginCityKey],* : 1,Single,Yes,Filters shipments by their origin city.
3,DimCity[CityKey],FactShipment[DestCityKey],* : 1,Single,No,Destination-city role; kept inactive to avoid ...
4,DimRoute[RouteKey],FactShipment[RouteKey],* : 1,Single,Yes,Filters shipments by route and corridor.
5,DimMode[ModeKey],FactShipment[ModeKey],* : 1,Single,Yes,Filters shipments by transport mode.
6,DimStatus[StatusKey],FactShipment[StatusKey],* : 1,Single,Yes,Filters shipments by delivery status.


In [3]:
# Required self-checks
assert len(relationships) == 6
assert (relationships["Cardinality"] == "* : 1").all()
assert (relationships["Cross-filter"] == "Single").all()
assert (relationships["Active?"] == "Yes").sum() == 5
assert (relationships["Active?"] == "No").sum() == 1
print("All five relationship-register self-checks pass.")

All five relationship-register self-checks pass.


## Part B · The two city roles

**Answer chosen: 1. Leave the second relationship inactive and activate it per measure with `USERELATIONSHIP`.**

The origin-city relationship is the active default because reports commonly filter shipments from their origin. The destination-city relationship remains inactive because `DimCity` cannot have both city-role relationships active to `FactShipment` without creating ambiguous filter paths. Destination-based measures can activate the inactive relationship explicitly with `USERELATIONSHIP`.

## Final reconciliation

Changing the relationship structure does not change the business totals: **578,650 kg** shipped and **PKR 4,808,000** in freight cost.